In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [2]:
orders      = pd.read_csv('../data/orders_dataset.csv')
order_items = pd.read_csv('../data/order_items_dataset.csv')
customers   = pd.read_csv('../data/customers_dataset.csv')

In [4]:
orders = orders[orders['order_status'] == 'delivered'].copy()
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


In [5]:
ingresos = order_items.groupby('order_id')['price'].sum().reset_index(name='monetary')
ingresos.head(3)

,order_id,monetary
0,00010242fe8c5a6d1ba2dd792cb16214,58.9
1,00018f77f2f0320c557190d7a144bdd3,239.9
2,000229ec398224ef6ca0657da4fc703e,199.0


In [7]:
df = orders.merge(ingresos, on='order_id', how='inner')
df = df.merge(customers[['customer_id', 'customer_unique_id']], on='customer_id', how='inner')
df.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,monetary,customer_unique_id
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,29.99,7c396fd4830fd04220f754e42b4e5bff
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,118.70,af07308b275d755c9edb36a90c618231
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,159.90,3a653a41f6f9fc3d2a113cf8398680e8


In [9]:
fecha_ref = orders['order_purchase_timestamp'].max() + pd.Timedelta(days=1)
fecha_ref

Timestamp('2018-08-30 15:00:37')

In [10]:
rfm = (
    df.groupby('customer_unique_id')
    .agg(
        recency  =('order_purchase_timestamp', lambda x: (fecha_ref - x.max()).days),
        frequency=('order_id', 'count'),
        monetary =('monetary', 'sum')
    )
    .reset_index()
)

print(f"Clientes totales en RFM: {len(rfm)}")
print(f"Fecha referencia recency: {fecha_ref.date()}")

Clientes totales en RFM: 93358
Fecha referencia recency: 2018-08-30


In [11]:
rfm.head(3)

,customer_unique_id,recency,frequency,monetary
0,0000366f3b9a7992bf8c76cfdf3221e2,112,1,129.9
1,0000b849f77a49e4a4ce2b2a4ca5be3f,115,1,18.9
2,0000f46a3911fa3c0805444483337064,537,1,69.0


In [13]:
print(rfm['frequency'].describe())
print(rfm['recency'].describe())
print(rfm['monetary'].describe())

count    93358.000000
mean         1.033420
std          0.209097
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         15.000000
Name: frequency, dtype: float64
count    93358.000000
mean       237.941773
std        152.591453
min          1.000000
25%        114.000000
50%        219.000000
75%        346.000000
max        714.000000
Name: recency, dtype: float64
count    93358.000000
mean       141.621480
std        215.694014
min          0.850000
25%         47.650000
50%         89.730000
75%        154.737500
max      13440.000000
Name: monetary, dtype: float64
